# AutoCategory AI — Text classification semester project

**Objective:** Predict fixed e-commerce product categories from title + description using TF–IDF features, compare several classical classifiers, explore a simple pseudo-labeling (self-training) step, and export artifacts compatible with the production FastAPI service (`model.pkl`, `vectorizer.pkl`, `label_encoder.pkl`).

**Run location:** The first code cell searches the current working directory and its parents for `train_model.py`, so starting Jupyter from the repo root, `ml-service/`, or `ml-service/notebooks/` all work.

## 1. Configuration and imports

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Resolve ml-service root: cwd, then parents (works from repo root, ml-service/, or notebooks/)
def _find_ml_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "train_model.py").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find train_model.py starting from cwd={start}. "
        "Cd into ml-service or open the project folder in Jupyter."
    )


_here = Path.cwd().resolve()
ML_ROOT = _find_ml_root(_here)
sys.path.insert(0, str(ML_ROOT))

import train_model as tm  # shared cleaning, paths, category normalization

ARTIFACTS_DIR = ML_ROOT / "models" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_MODEL_DIR = ML_ROOT / "model"
EXPORT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
sns.set_theme(style="whitegrid")
print("ML_ROOT:", ML_ROOT)
print("Artifacts:", ARTIFACTS_DIR)

## 2. Load and merge datasets

Uses the same column aliasing and normalization as `train_model.py`. Pass explicit CSV paths via `DATA_FILES` or leave `None` to auto-detect defaults under `ml-service/data/`.

In [ ]:
# Set to e.g. ["data/my_products.csv"] or None for train_model defaults
DATA_FILES = None

paths = tm.resolve_dataset_paths(DATA_FILES)
merged = tm.load_and_merge_datasets(paths)
raw_count = len(merged)
cleaned = tm.prepare_clean_dataset(merged)
print(f"Rows raw: {raw_count}, after cleaning: {len(cleaned)}")
cleaned.head()

## 3. EDA — class distribution

In [ ]:
counts = cleaned["category"].value_counts().sort_index()
ax = counts.plot(kind="bar", figsize=(10, 4), color="steelblue", edgecolor="black")
ax.set_title("Category distribution (normalized labels)")
ax.set_xlabel("Category")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
counts

## 4. Train / test split and label encoding

Text field: `cleaned_text` (title + description), same as the training script.

In [ ]:
texts = cleaned["cleaned_text"].tolist()
labels = cleaned["category"].tolist()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

stratify = y if (pd.Series(y).value_counts() >= 2).all() else None
x_train, x_test, y_train, y_test = train_test_split(
    texts, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
)
print("Train:", len(x_train), "Test:", len(x_test))

## 5. TF–IDF + model training (four classifiers)

Fixed vectorizer settings for a fair comparison (same split, same features). Production training may additionally tune hyperparameters via `python train_model.py` (GridSearchCV).

In [ ]:
def build_pipeline(model) -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2),
            ),
            ("model", model),
        ]
    )


def eval_split(name: str, pipe: Pipeline) -> dict:
    pipe.fit(x_train, y_train)
    pred = pipe.predict(x_test)
    return {
        "model_name": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, pred, average="weighted", zero_division=0),
        "pipeline": pipe,
        "y_pred": pred,
    }


candidates = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ("MultinomialNB", MultinomialNB()),
    ("LinearSVC", LinearSVC(random_state=RANDOM_STATE)),
    (
        "RandomForest",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced_subsample",
        ),
    ),
]

results = [eval_split(name, build_pipeline(model)) for name, model in candidates]
compare = (
    pd.DataFrame([{k: r[k] for k in r if k not in {"pipeline", "y_pred"}} for r in results])
    .sort_values("f1_score", ascending=False)
    .reset_index(drop=True)
)
compare

## 6. Metrics comparison table (export CSV / JSON)

In [ ]:
compare_path = ARTIFACTS_DIR / "notebook_model_comparison.csv"
compare.to_csv(compare_path, index=False)
metrics_path = ARTIFACTS_DIR / "notebook_metrics.json"
metrics_path.write_text(
    json.dumps(
        {"random_state": RANDOM_STATE, "test_size": TEST_SIZE, "models": compare.to_dict(orient="records")},
        indent=2,
    ),
    encoding="utf-8",
)
print("Wrote", compare_path)
print("Wrote", metrics_path)

## 7. Confusion matrix (best model by weighted F1)

In [ ]:
best = max(results, key=lambda r: r["f1_score"])
class_names = [str(c) for c in label_encoder.classes_]
cm = confusion_matrix(y_test, best["y_pred"], labels=list(range(len(class_names))))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix — {best['model_name']}")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(y_test, best["y_pred"], target_names=class_names, zero_division=0))

## 8. Semi-supervised experiment (pseudo-labeling)

Reuses `run_semi_supervised_self_training` from `train_model.py`: train on a subset of the training pool, add high-confidence pseudo-labels from the rest, retrain, and compare **test** metrics before vs after.

In [ ]:
PSEUDO_CONFIDENCE = 0.92

semi = tm.run_semi_supervised_self_training(
    x_train=list(x_train),
    y_train=np.asarray(y_train),
    x_test=list(x_test),
    y_test=np.asarray(y_test),
    label_encoder=label_encoder,
    best_pipeline_template=best["pipeline"],
    random_state=RANDOM_STATE,
    confidence_threshold=PSEUDO_CONFIDENCE,
)
semi_out = ARTIFACTS_DIR / "notebook_semi_supervised_metrics.json"
semi_out.write_text(json.dumps(semi, indent=2), encoding="utf-8")
print(json.dumps(semi, indent=2))
print("Wrote", semi_out)

## 9. Export production artifacts (optional)

This matches what FastAPI loads: separate estimator and vectorizer files. For **tuned** hyperparameters and full grid comparison, prefer the CLI: `python train_model.py ...`.

In [ ]:
best_pipe = best["pipeline"]
joblib.dump(best_pipe.named_steps["model"], EXPORT_MODEL_DIR / "model.pkl")
joblib.dump(best_pipe.named_steps["tfidf"], EXPORT_MODEL_DIR / "vectorizer.pkl")
joblib.dump(label_encoder, EXPORT_MODEL_DIR / "label_encoder.pkl")

info = {
    "schema_version": 1,
    "best_classifier": best["model_name"],
    "notebook_export": True,
    "num_labels": int(len(label_encoder.classes_)),
    "labels": [str(c) for c in label_encoder.classes_],
    "test_metrics": {k: float(best[k]) for k in ["accuracy", "precision", "recall", "f1_score"]},
}
(EXPORT_MODEL_DIR / "model_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
(EXPORT_MODEL_DIR / "class_labels.json").write_text(
    json.dumps([str(c) for c in label_encoder.classes_], indent=2), encoding="utf-8"
)
print("Exported to", EXPORT_MODEL_DIR)

## 10. Conclusion

- **Baseline comparison:** Four sklearn classifiers were trained on the same TF–IDF features and test split; the comparison table is saved under `models/artifacts/`.
- **Semi-supervised:** Pseudo-labeling shows whether adding high-confidence self-labels on a held-out portion of the training pool improves generalization on the fixed test set (see `notebook_semi_supervised_metrics.json`).
- **Production:** Run `python train_model.py` for GridSearch-tuned models, `model_comparison.csv`, `metrics.json`, and optional `semi_supervised_metrics.json`; deploy pickles + `model_info.json` to the server `model/` directory and restart `autocategory-fastapi` (see `TRAINING_AND_DEPLOYMENT.md`).

Mobile / Laravel clients keep using `POST /predict` unchanged; `GET /model-info` is optional metadata only.